#### Task 1: Simple vector embedding generation

**Objective:**
Generate vector embeddings from text data.

**Task Description:**

- load embedding model into ollama:
    1. Attach ollama container shell
    2. Run command in container `ollama pull pull bge-m3` or send an api call `requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False}` via the requests library to download embedding model.
- embed simple text queries

How to select the right embedding model: [MTEB - Massive Text Embedding Benchmark](https://huggingface.co/blog/mteb)

**Useful links:**

- [Ollama REST API](https://www.postman.com/postman-student-programs/ollama-api/documentation/suc47x8/ollama-rest-api)
- [Langchain Ollama Embeddings](https://python.langchain.com/docs/integrations/text_embedding/ollama/)
- [Langchain Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/)


In [2]:
import requests

In [3]:
# Download embedding model into ollama container
requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False})

<Response [200]>

In [4]:
# List all available models in the ollama container
response = requests.get("http://localhost:11434/api/ps")
response.json()

{'models': []}

In [ ]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODEs
embedding_model = OllamaEmbeddings(model = "bge-m3")

In [7]:
text = "This is a test document."

# ADD HERE YOUR CODE
# Perform vector search
query_vector = embedding_model.embed_query(text)

print(f"Embedding vector length: {len(query_vector)}")
print(query_vector[:10])

ResponseError: model 'llama3' not found (status code: 404)

#### Task 2: Generate embedding vectors with custom dataset

**Objective:**
Load custom dataset, preprocess it and generate vector embeddings.

**Task Description:**

- load pdf document "AI_Book.pdf" via langchain document loader: ``PyPDFLoader``
- use RecursiveCharacterTextSplitter to split documents into chunks
- generate embeddings for single documents

**RecursiveCharacterTextSplitter:**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

**Useful links:**

- [Langchain PyPDFLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/)
- [Langchain RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

pdf_doc = "./AI_Book.pdf"

# Create pdf data loader
# ADD HERE YOUR CODE
loader = ...

# Load and split documents in chunks
# ADD HERE YOUR CODE
pages_chunked = ...

print(pages_chunked[0])

# Function to clean text by removing invalid unicode characters, including surrogate pairs
def clean_text(text):
    # Remove surrogate pairs
    text = re.sub(r'[\ud800-\udfff]', '', text)
    # Optionally remove non-ASCII characters (depends on your use case)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

pages_chunked_cleaned = [
    Document(page_content=clean_text(doc.page_content), metadata=doc.metadata)
    for doc in pages_chunked
]

print(pages_chunked_cleaned[0])


In [ ]:
print(pages_chunked_cleaned[1])

In [ ]:
print(f"Number of text chunks: {len(pages_chunked_cleaned)}")

#### Task 3: Store vector embeddings from pdf document to ChromaDB vector database.

**Objective:**

Store vector embeddings into ChromaDB to store knowledge.

**Task Description:**

- create chromadb client
- create chromadb collection
- create langchain chroma db client
- store text document chunks and vector embeddings to vector databases

**Useful links:**

- [Langchain How To](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#initialization-from-client)

In [ ]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

collection_name = "ai_model_book"

# ADD HERE YOUR CODE
# Create a collection
collection = ...

# ADD HERE YOUR CODE
# Create chromadb
vector_db_from_client = Chroma(...)

In [ ]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(pages_chunked_cleaned))]

# ADD HERE YOUR CODE
vector_db_from_client.add_documents(...) # that can take a while...

In [9]:
client.count_collections()

1

In [89]:
# client.delete_collection("ai_model_book")

In [ ]:
# Get and check the content of the collection
collection.get(include=["documents", "metadatas", "embeddings"])

#### Task 4: Access ChromaDB and perform vector search

**Objective:**

Use query to perform vector search against ChromaDB vector database

**Task Description:**

- define query
- run vector search
- print k=3 most similar documents


**Useful links:**

- [Langchain Query ChromaDB](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#search-by-vector)

In [10]:
search_query = "Types of Machine Learning Systems"

results = vector_db_from_client.similarity_search(...)

for res in results:
    print(res.page_content)
    print(res.metadata)
    print("\n")